# Machine Learning Intern - Assignment Project
Company: ServiceHive Product: Inflx Project Title: Social-to-Lead Agentic Workflow

ServiceHive is building Inflx, an AI-powered platform that converts social media conversations into qualified business leads.
Unlike simple chatbots, Inflx agents are designed to:
1. Understand user intent
2. Answer product questions accurately
3. Identify high-intent users
4. Trigger backend actions like lead capture

## Install Dependencies

## Step 1: Intent Detection

In [1]:
def detect_intent(text):
    text = text.lower()

    if "hi" in text or "hello" in text:
        return "greeting"

    elif "buy" in text or "subscribe" in text or "want" in text or "try" in text:
        return "high_intent"

    else:
        return "pricing"

In [2]:
print(detect_intent("Hi"))
print(detect_intent("What is price?"))
print(detect_intent("I want to buy"))
print(detect_intent("I want to buy Pro plan"))

greeting
pricing
high_intent
high_intent


## Step 2: Loading local knowledge base (RAG) Json file

In [5]:
import json

with open("local_knowledge_base(RAG).json") as f:
    data = json.load(f)

## Step 3: RAG Response

In [6]:
def rag_response(query):
    query = query.lower()

    # Basic plan details
    if "basic" in query:
        return f"""
Basic Plan:
Price: {data['basic_plan']['price']}
Features: {", ".join(data['basic_plan']['features'])}
"""

    # Pro plan details
    elif "pro" in query:
        return f"""
Pro Plan:
Price: {data['pro_plan']['price']}
Features: {", ".join(data['pro_plan']['features'])}
"""

    # Pricing overview
    elif "price" in query or "plan" in query:
        return f"""
Basic: {data['basic_plan']['price']}
Pro: {data['pro_plan']['price']}
"""

    # Features
    elif "feature" in query:
        return f"""
Basic Features: {", ".join(data['basic_plan']['features'])}
Pro Features: {", ".join(data['pro_plan']['features'])}
"""

    # Policies (handles policy + policies)
    elif "polic" in query or "refund" in query:
        return "\n".join(data["policies"])

    # Default fallback
    else:
        return "Please ask about pricing, features, or policies."

In [7]:
print(rag_response("What is the price?"))
print(rag_response("Tell me about pro plan"))
print(rag_response("What is refund policy?"))
print(rag_response("What are the features?"))


Basic: $29/month
Pro: $79/month


Pro Plan:
Price: $79/month
Features: Unlimited videos, 4K resolution, AI captions

No refunds after 7 days
24/7 support only for Pro plan

Basic Features: 10 videos/month, 720p resolution
Pro Features: Unlimited videos, 4K resolution, AI captions



## Step 4: Lead Capture Tool

In [8]:
def mock_lead_capture(name, email, platform):
    print(f"Lead captured successfully: {name}, {email}, {platform}")

In [9]:
lead_data = {"name": None,"email": None,"platform": None}

conversation_state = {"stage": "start"}

## Step 5: Agent Workflow

In [10]:
def is_valid_email(email):
    if "@" in email and "." in email:
        return True
    return False

In [11]:
allowed_platforms = ["youtube", "instagram", "tiktok", "facebook", "linkedin"]

In [12]:
def agent(user_input):

    text = user_input.lower()

    # 1. Lead collection flow FIRST (priority)
    if conversation_state["stage"] == "ask_name":
        lead_data["name"] = user_input
        conversation_state["stage"] = "ask_email"
        return "Enter your email"

    elif conversation_state["stage"] == "ask_email":
        if not is_valid_email(user_input):
            return "Please enter a valid email (example: name@gmail.com)"

        lead_data["email"] = user_input
        conversation_state["stage"] = "ask_platform"
        return "Which platform do you use? (YouTube, Instagram, TikTok, Facebook, LinkedIn)"

    elif conversation_state["stage"] == "ask_platform":
        allowed_platforms = ["youtube", "instagram", "tiktok", "facebook", "linkedin"]

        if text not in allowed_platforms:
            return "Please choose from: YouTube, Instagram, TikTok, Facebook, LinkedIn"

        lead_data["platform"] = user_input

        mock_lead_capture(
            lead_data["name"],
            lead_data["email"],
            lead_data["platform"]
        )

        conversation_state["stage"] = "post_lead"
        return "Thanks! We will contact you soon. Do you have any more queries? (yes/no)"

    elif conversation_state["stage"] == "post_lead":
        if text == "yes":
            conversation_state["stage"] = "start"
            return "Sure! Ask me anything about pricing or plans."
        else:
            conversation_state["stage"] = "done"
            return "Thank you! Have a great day."

    # 2. Handle "not interested" BEFORE intent
    if text in ["no", "nope", "not interested"]:
        conversation_state["stage"] = "done"
        return "No problem! Thank you for your time."

    # 3. Intent detection
    intent = detect_intent(user_input)

    if intent == "greeting":
        return "Hello! Ask me about pricing or plans."

    elif intent == "pricing":
        return rag_response(user_input)

    elif intent == "high_intent":
        conversation_state["stage"] = "ask_name"
        return "Great! What's your name?"

    return "Please ask about pricing, plans, or policies."

In [13]:
conversation_state["stage"] = "start"
lead_data = {"name": None, "email": None, "platform": None}

## Step 6: Test Conversation

In [14]:
while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    response = agent(user)
    print("Bot:", response)

    # End only if user says no
    if conversation_state["stage"] == "done":
        break

You:  hi


Bot: Hello! Ask me about pricing or plans.


You:  plans


Bot: 
Basic: $29/month
Pro: $79/month



You:  i want to know about the plans


...

Bot: Please choose from: YouTube, Instagram, TikTok, Facebook, LinkedIn


You:  instagram


Lead captured successfully: khsdf, jh@yahoo.in, instagram
Bot: Thanks! We will contact you soon. Do you have any more queries? (yes/no)


You:  no


Bot: Thank you! Have a great day.


## Insights

- Demonstrates an agentic AI workflow for handling user interactions
- Uses rule-based intent detection (can be extended using LLMs)
- Implements RAG using a local JSON knowledge base
- Executes tool (lead capture) only after complete data collection
- Maintains conversation state across multiple turns

### Improvements
- Replace rule-based intent detection with LLM-based classification for better accuracy
- Implement vector database (e.g., FAISS) for scalable RAG instead of keyword matching
- Integrate with frameworks like LangChain or LangGraph for better workflow orchestration
- Deploy the agent using Streamlit or integrate with WhatsApp API using webhooks